In [3]:
import lightkurve as lk
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from astropy.timeseries import LombScargle
import astropy.units as u
import gyrointerp
from gyrointerp import gyro_age_posterior
from gyrointerp import get_summary_statistics

targets = pd.read_csv("C:\\Users\\smithlt\\Documents\\ASTR502\\TOI-431_with_tars.csv")

In [4]:
print(targets.head())

      TICID        dr2_source_id        dr3_source_id          ra        dec  \
0   5667848  3035176873248454656  3035176873248454656  111.656735 -10.954077   
1   7221735  4943207710712867712  4943207710712867712   32.485480 -45.773100   
2  11285126  3215687098789011456  3215687098789011456   84.639885  -3.907087   
3  14612845  3163115462135824256  3163115462135824256  111.376373  12.471692   
4  21650901  2887048811323390848  2887048811323390848   87.625427 -36.939414   

         pmra       pmdec   parallax         teff     Tmag  ...  W1-W3   RUWE  \
0 -118.897143   53.092510  23.262654  3265.837240  12.9095  ...    NaN  1.170   
1  289.004228  142.086380  32.287421  3547.650553  10.8302  ...    NaN  1.228   
2    2.543500   10.068257  19.391355  3122.914711  14.3243  ...    NaN  1.068   
3 -145.181953  -65.124441  28.662080  3091.807717  13.7338  ...    NaN  1.137   
4   -0.434907  135.324024  21.870991  3215.886859  13.1768  ...    NaN  1.143   

   XCrate     RVsrc  PMRApred  P

In [7]:
print(f"Loaded {len(targets)} total targets from CSV")
print(f"Columns available: {list(targets.columns)}")

#create a new dataframe that contains the relevant columns for age determination
targets_df = targets[['TICID', 'teff', 'adopted_period', 'adopted_period_unc']]
print(targets_df.head())

Loaded 29 total targets from CSV
Columns available: ['TICID', 'dr2_source_id', 'dr3_source_id', 'ra', 'dec', 'pmra', 'pmdec', 'parallax', 'teff', 'Tmag', 'phot_g_mean_mag', 'phot_g_mean_mag_0', 'phot_bp_mean_mag', 'phot_bp_mean_mag_0', 'phot_rp_mean_mag', 'phot_rp_mean_mag_0', 'BpmRp0', 'extinction_a0', 'ruwe', 'non_single_star', 'adopted_period', 'adopted_period_unc', 'flag_multiple_periods', 'flag_possible_binary', 'final_n_contams', 'flag_doubled_period', 'n_secs', 'n_sec_ratio', 'median_amplitude', 'sectors', 'sector_periods', 'sector_sys_probs', 'sector_sig_probs', 'sector_match_probs', 'sector_alias_probs', 'sector_amplitudes', 'Catalog', 'Type', 'RA', 'DEC', 'Gmag', 'Bp-Rp', 'Voff(km/s)', 'Sep(deg)', '3D(pc)', 'Vr(pred)', 'Vr(obs)', 'Vrerr', 'Plx(mas)', 'SpT', 'FnuvJ', 'W1-W3', 'RUWE', 'XCrate', 'RVsrc', 'PMRApred', 'PMDecpred', 'PMRA', 'PMRAerr', 'PMDec', 'PMDecerr']
      TICID         teff  adopted_period  adopted_period_unc
0   5667848  3265.837240        4.952409           

In [8]:
Teff = targets_df['teff']
Prot = targets_df['adopted_period']
Prot_unc = targets_df['adopted_period_unc']

In [ ]:
#cycle through all of the targets and run gyrointerp
#print the age posterior calculated for each target

for i in range(len(targets_df)):

    # units: days
    Prot, Prot_err = targets_df['adopted_period'].iloc[i], targets_df['adopted_period_unc'].iloc[i]
    print(Prot)

    # units: kelvin
    Teff, Teff_err = targets_df['teff'].iloc[i], 100
    print(Teff)

    # uniformly spaced grid between 0 and 4000 megayears
    age_grid = np.linspace(0, 4000, 500)

    # calculate the age posterior at each age in `age_grid`
    age_posterior = gyro_age_posterior(
        Prot, Teff,
        Prot_err=Prot_err, Teff_err=Teff_err,
        age_grid=age_grid
    )

    print(f"\nTarget: {targets_df['TICID'].iloc[i]}")
    print(f"Age posterior (normalized): {age_posterior / np.sum(age_posterior)}")
    print(f"Age (most probable): {age_grid[np.argmax(age_posterior)]:.1f} Myr")

4.952409141006385
3265.8372396662644

Target: 5667848
Age posterior (normalized): [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan n

[W 260331 11:21:30 gyro_posterior:365] WARNING: imposing period uncertainty floor of 0.03d The purpose of this floor is to prevent N_grid from growing too large.  This is justified because the gyro model has no information content on this scale.



Target: 71628926
Age posterior (normalized): [3.60577587e-12 3.60577587e-12 3.60577587e-12 3.60577587e-12
 3.60577587e-12 3.60577587e-12 3.60577587e-12 3.60577587e-12
 3.60577587e-12 3.60577587e-12 3.83019737e-12 7.76508665e-11
 1.27695483e-09 1.36546100e-08 7.82440408e-08 2.15978296e-07
 3.96239467e-07 6.82492419e-07 1.10761537e-06 1.71805056e-06
 2.54036265e-06 3.59179669e-06 4.88863714e-06 6.38726030e-06
 8.10823981e-06 1.00513764e-05 1.21424441e-05 1.43882775e-05
 1.69319280e-05 1.97315987e-05 2.28609926e-05 2.64013024e-05
 3.04656939e-05 3.52027141e-05 4.06934226e-05 4.74578308e-05
 5.57895238e-05 6.61955706e-05 7.92204794e-05 9.45508493e-05
 1.12788861e-04 1.34424396e-04 1.59828306e-04 1.89568844e-04
 2.24065816e-04 2.63892311e-04 3.10238674e-04 3.63753984e-04
 4.25007915e-04 4.94417115e-04 5.73988494e-04 6.63437695e-04
 7.64730197e-04 8.77992548e-04 1.00472690e-03 1.14404978e-03
 1.29890896e-03 1.47054150e-03 1.65844614e-03 1.86210456e-03
 2.08407843e-03 2.32172379e-03 2.578397

In [ ]:
# calculate dictionary of summary statistics for each target and store results in targets_df
# Note: gyro_age_posterior and get_summary_statistics were imported in earlier cells,
# so we don't re-import them here.

# ensure columns exist (store arrays as objects)
for col in ['age_grid', 'age_posterior', 'median', '+1sigma', '-1sigma', 'mean', 'mode']:
    if col not in targets_df.columns:
        targets_df[col] = [None] * len(targets_df)

for i in range(targets_df.shape[0]):
    Prot = targets_df['adopted_period'].iloc[i]
    Prot_err = targets_df['adopted_period_unc'].iloc[i]

    Teff = targets_df['teff'].iloc[i]
    Teff_err = 100

    # uniformly spaced grid between 0 and 4000 megayears
    age_grid = np.linspace(0, 4000, 500)

    # calculate the age posterior at each age in `age_grid`
    age_posterior = gyro_age_posterior(
        Prot, Teff,
        Prot_err=Prot_err, Teff_err=Teff_err,
        age_grid=age_grid
    )

    # compute summary statistics
    result = get_summary_statistics(age_grid, age_posterior)

    # store results in the dataframe
    targets_df.at[i, 'age_grid'] = age_grid
    targets_df.at[i, 'age_posterior'] = age_posterior
    targets_df.at[i, 'median'] = result.get('median', np.nan)
    targets_df.at[i, '+1sigma'] = result.get('+1sigma', np.nan)
    targets_df.at[i, '-1sigma'] = result.get('-1sigma', np.nan)
    targets_df.at[i, 'mean'] = result.get('mean', np.nan)
    targets_df.at[i, 'mode'] = result.get('mode', np.nan)

    print(f"\nTarget: {targets_df['TICID'].iloc[i]}")
    print(f"Age = {result['median']} +{result['+1sigma']} -{result['-1sigma']} Myr.")
